# 01 — Tokenization: Turning Text Into Numbers

**Lecture goal:** understand *why* text has to become numbers before a neural network can touch it, and build the simplest possible thing that does that conversion.

### The big picture

A large language model is, underneath everything, a function: it takes in a sequence of numbers and produces a probability distribution over what number should come next. It never sees the letter `"a"` or the word `"cat"` — it only ever sees integers.

So before we can write a single line of the model itself, we need a pipeline that goes:

```
"Once upon a time"  --->  [1, 245, 3, 89]  --->  (later: back to text again)
     raw text              token IDs
```

The process of splitting raw text into a sequence of discrete units ("tokens") is called **tokenization**. The process of assigning each unique token an integer is building a **vocabulary**. Today we build both from scratch using plain Python — no PyTorch needed yet. PyTorch shows up in notebook 03, once we need to feed batches of these numbers into a model efficiently.

We'll use a real, small piece of text to work with: `the-verdict.txt`, a short story by Edith Wharton, sitting right next to this notebook.

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:200])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


## Step 1: The naive approach — split on whitespace

The simplest thing we could do is split the text every time we see a space, using Python's built-in `str.split()`. Let's try it on a short example and see where it breaks down.

In [2]:
sample = "Hello, world. Is this-- a test?"

words = sample.split()
print(words)

['Hello,', 'world.', 'Is', 'this--', 'a', 'test?']


Notice the problem: `"world."` and `"test?"` still have punctuation glued onto them. If `"world"` and `"world."` are treated as two completely different tokens, the model has to separately learn everything about both — a huge waste. We want punctuation split out as its own tokens.

## Step 2: Splitting with regular expressions

A **regular expression** (regex) is a mini pattern-language for matching pieces of text. We'll use Python's `re` module and `re.split(pattern, text)`, which splits `text` wherever `pattern` matches — similar to `str.split()`, but the pattern can be far more expressive than "just spaces".

The pattern we want: split on whitespace, and *also* split on punctuation characters, but keep the punctuation itself around (don't throw it away).

`re.split` normally discards whatever it matched. To keep the delimiters, wrap the pattern in parentheses `( ... )` — this turns it into a "capture group", and `re.split` includes captured text in its output.

In [3]:
import re

text = "Hello, world. Is this-- a test?"

# Split on: comma, period, whitespace — keep them in the output via ( )
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'Is', ' ', 'this--', ' ', 'a', ' ', 'test?']


Better — punctuation is now separate from words. But there are two remaining problems:

1. Plain whitespace tokens (`' '`) are floating around, which we don't need as tokens themselves.
2. We're only handling commas and periods. Real text has `--`, `"`, `'`, `;`, `:`, `?`, `!`, `(`, `)` and more.

Let's fix both: extend the pattern to cover more punctuation, then filter out whitespace-only entries.

In [4]:
def tokenize(text):
    # Split on any of: -- , . : ; ? _ ! " ' ( ) or whitespace — keeping the punctuation.
    split_pattern = r'([,.:;?_!"()\']|--|\s)'
    raw_tokens = re.split(split_pattern, text)
    # Drop empty strings and pieces that are just whitespace.
    tokens = [tok.strip() for tok in raw_tokens if tok.strip()]
    return tokens

print(tokenize("Hello, world. Is this-- a test?"))

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


This `tokenize` function is a reasonable, simple tokenizer. Let's run it on the entire short story and see how many tokens we get.

In [5]:
all_tokens = tokenize(raw_text)

print("Number of tokens:", len(all_tokens))
print(all_tokens[:30])

Number of tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Step 3: Building a vocabulary

We now have a list of token *strings*. A neural network needs *integers*. So we build a **vocabulary**: a mapping from each unique token string to a unique integer ID.

The recipe:
1. Collect the *unique* tokens (`set(...)`).
2. Sort them, so the mapping is deterministic and reproducible.
3. Enumerate them: the first (alphabetically) token gets ID `0`, the next gets `1`, and so on.

In [6]:
unique_tokens = sorted(set(all_tokens))
vocab_size = len(unique_tokens)
print("Vocabulary size:", vocab_size)

vocab = {token: integer for integer, token in enumerate(unique_tokens)}

# Peek at the first 10 entries of the vocabulary
for i, (token, integer) in enumerate(vocab.items()):
    print(token, "->", integer)
    if i >= 9:
        break

Vocabulary size: 1130
! -> 0
" -> 1
' -> 2
( -> 3
) -> 4
, -> 5
-- -> 6
. -> 7
: -> 8
; -> 9


## Step 4: Wrapping this into a reusable tokenizer class

Rather than calling loose functions, let's package encode (text → IDs) and decode (IDs → text) into one class, `SimpleTokenizerV1`. This mirrors how real tokenizers (and the `tiktoken` library we'll use in notebook 02) are structured: build the vocabulary once, then reuse it via `.encode()` / `.decode()`.

In [7]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer: token for token, integer in vocab.items()}

    def encode(self, text):
        preprocessed = tokenize(text)
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        # Remove the space PyTorch/we inserted before punctuation, e.g. "Hello , world" -> "Hello, world"
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [8]:
tokenizer = SimpleTokenizerV1(vocab)

sample_text = """"It\'s the last he painted, you know," Mrs. Gisburn said."""
ids = tokenizer.encode(sample_text)
print("IDs:   ", ids)
print("Decoded:", tokenizer.decode(ids))

IDs:    [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 7]
Decoded: " It' s the last he painted, you know," Mrs. Gisburn said.


Round trip works: text → IDs → back to (almost) the original text. This is the essence of every tokenizer we'll use going forward, including the professional-grade one in the next notebook.

### The problem: what about words we've never seen?

Our vocabulary only contains words that appeared in `the-verdict.txt`. What happens if we try to encode a word that isn't in there — like `"Hello"`, which never appears in this particular short story?

In [9]:
text = "Hello, do you like tea?"
try:
    print(tokenizer.encode(text))
except KeyError as e:
    print(f"KeyError: {e} is not in the vocabulary!")

KeyError: 'Hello' is not in the vocabulary!


That raises a `KeyError` — and it's supposed to. `"Hello"` was never added to `str_to_int`, so there's no integer for it to map to. This is a fundamental limitation of a fixed, closed vocabulary: **any word outside the training text is simply impossible to represent.**

Real systems need a way to say "I don't recognize this token, but I still need to represent *something* here." That's what **special tokens** are for.

## Step 5: Special tokens

We'll add two special, reserved tokens to the vocabulary:

- `<|unk|>` — stands in for any word not in our vocabulary ("unknown").
- `<|endoftext|>` — marks a boundary between two separate, unrelated documents. When we later train on many documents concatenated together, the model needs a signal that says "the previous document has ended; do not treat what follows as a continuation of it."

Both get added to the vocabulary just like any other token, each with their own integer ID.

### Why the document boundary matters so much

`<|unk|>` is easy to motivate; `<|endoftext|>` deserves a moment. Training text never comes from one tidy source. A realistic corpus is a pile of unrelated material stitched together — news articles, then Reddit threads, then blog posts, then scanned newspaper clippings — and once tokenized it's just one long, undifferentiated stream of IDs. Without a marker, the model would read the last sentence of a news article and the first sentence of a blog post as a single continuous passage, and dutifully try to learn the "transition" between them.

So between every pair of unrelated sources we insert `<|endoftext|>`. It acts as a seam: *this segment is finished, what follows is unrelated.* GPT's own training data was prepared exactly this way, and the token survives into inference — it's how you signal "stop" or separate independent prompts.

### Other special tokens you'll encounter

`<|unk|>` and `<|endoftext|>` aren't the whole universe. Depending on the model and the task you'll also see tokens for the beginning of a sequence (`[BOS]`), the end of a sequence (`[EOS]`), and padding (`[PAD]`, used to bring short sequences in a batch up to a common length). We won't need any of them here, but it's worth knowing the vocabulary of special tokens is a design choice, not a fixed list.

And a preview of the next notebook: GPT's tokenizer uses `<|endoftext|>` but has **no `<|unk|>` token at all.** That isn't an oversight — the subword scheme we're about to meet makes unknown words impossible in the first place.

In [10]:
all_tokens_extended = sorted(set(all_tokens))
all_tokens_extended.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens_extended)}
print("New vocabulary size:", len(vocab))

# Confirm the special tokens landed at the end
for token, integer in list(vocab.items())[-5:]:
    print(token, "->", integer)

New vocabulary size: 1132
younger -> 1127
your -> 1128
yourself -> 1129
<|endoftext|> -> 1130
<|unk|> -> 1131


Now `SimpleTokenizerV2`: identical to `V1`, except `encode` replaces any token that isn't in the vocabulary with `<|unk|>` instead of raising an error.

In [11]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer: token for token, integer in vocab.items()}

    def encode(self, text):
        preprocessed = tokenize(text)
        preprocessed = [
            token if token in self.str_to_int else "<|unk|>"
            for token in preprocessed
        ]
        ids = [self.str_to_int[token] for token in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [12]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

print(text)
print()

ids = tokenizer.encode(text)
print("IDs:", ids)
print()
print("Decoded:", tokenizer.decode(ids))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

IDs: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

Decoded: <|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


`"Hello"` — a genuinely unknown word — became `<|unk|>` instead of crashing the program, and `<|endoftext|>` correctly marks the seam between the two unrelated sentences.

## Recap

- Raw text needs to become integers before any neural network can process it.
- A **tokenizer** splits text into tokens (here: words and punctuation via regex); a **vocabulary** maps each unique token to an integer ID.
- A closed, fixed vocabulary has a hard limitation: unseen words are simply impossible to encode. `<|unk|>` patches this; `<|endoftext|>` marks the seams between unrelated documents in a corpus that was stitched together from many sources.
- Which special tokens exist is a design decision — `[BOS]`, `[EOS]` and `[PAD]` are common elsewhere, and GPT's own tokenizer keeps `<|endoftext|>` while dropping `<|unk|>` entirely.

### What's next

Our word-level vocabulary has a second, sneakier problem: it can only ever contain whole words we happened to see during "training" of the vocabulary itself. Any *typo*, rare word, or word in another language still becomes `<|unk|>` — an enormous loss of information. Real LLMs solve this with **subword tokenization**, most commonly **Byte Pair Encoding (BPE)**, which we'll build an intuition for — and use a production implementation of — in notebook 02.